# Okaí: 2. Análise das avaliações reais

**Entrada:** `2w3kvrg97m-1.zip`, com 730.069 avaliações de hotéis em português, publicadas no Mendeley Data (Brazilian Portuguese Hotel's Reviews Corpus, licença CC BY 4.0).

**Saída:** `modelo_sentimento.json` e `resumo_avaliacoes.json`, usados no site.

**Antes de rodar:** clique no ícone de pasta, à esquerda, e arraste o arquivo `.zip` para lá. Espere o envio terminar. Depois use *Ambiente de execução > Executar tudo*. O processo pode levar alguns minutos.

## 1. Abrir os arquivos
O arquivo traz um `.zip` dentro de outro, com quatro listas alinhadas: comentário, título, nota e data.

In [ ]:
import glob, io, json, re, unicodedata, zipfile
import numpy as np
import pandas as pd

ARQUIVO = "2w3kvrg97m-1.zip"

with zipfile.ZipFile(ARQUIVO) as externo:
    interno = [n for n in externo.namelist() if n.endswith(".zip")][0]
    with zipfile.ZipFile(io.BytesIO(externo.read(interno))) as z:
        def ler(nome):
            return z.read(nome).decode("utf-8").replace("\r", "").split("\n")
        comentarios, notas, datas = ler("commentsBrazilNormalized.txt"), ler("gradesBrazil.txt"), ler("datesBrazil.txt")

print(len(comentarios), len(notas), len(datas))
df = pd.DataFrame({"texto": comentarios, "nota": notas, "data": datas})
df["nota"] = pd.to_numeric(df["nota"], errors="coerce")
df["ano"] = pd.to_datetime(df["data"], format="%d/%m/%Y", errors="coerce").dt.year
df = df[df["nota"].between(1, 5) & (df["texto"].str.strip() != "")].drop_duplicates("texto")
print(f"Avaliações válidas e sem repetição: {len(df):,}".replace(",", "."))
df["nota"].value_counts().sort_index()

## 2. Criar o sentimento a partir da nota
O modelo aprende a partir das notas dadas pelas próprias pessoas:

- **1 ou 2:** negativo
- **3:** neutro
- **4 ou 5:** positivo

In [ ]:
df["sentimento"] = df["nota"].map({1: "negativo", 2: "negativo", 3: "neutro", 4: "positivo", 5: "positivo"})
df["sentimento"].value_counts(normalize=True).round(3)

## 3. Treinar e testar o modelo
Separamos 20% das avaliações para o teste. O modelo não vê esses textos durante o treino, então o resultado mostra como ele lida com comentários novos.

Usamos TF-IDF, que transforma palavras e pares de palavras em números, e regressão logística, um modelo simples e fácil de explicar. Como a maioria das avaliações é positiva, o treino dá mais peso às classes menores.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    df["texto"], df["sentimento"], test_size=0.2, random_state=42, stratify=df["sentimento"])

vetor = TfidfVectorizer(lowercase=True, strip_accents="unicode", ngram_range=(1, 2),
                        min_df=5, max_features=150_000, sublinear_tf=True)
A_treino = vetor.fit_transform(X_treino)
A_teste = vetor.transform(X_teste)

modelo = LogisticRegression(max_iter=300, class_weight="balanced", C=1.0)
modelo.fit(A_treino, y_treino)
previsto = modelo.predict(A_teste)

acuracia = accuracy_score(y_teste, previsto)
f1_macro = f1_score(y_teste, previsto, average="macro")
print(f"Acurácia: {acuracia:.3f}   F1 macro: {f1_macro:.3f}")
print(classification_report(y_teste, previsto, digits=3))
classes = list(modelo.classes_)
pd.DataFrame(confusion_matrix(y_teste, previsto, labels=classes), index=classes, columns=classes)

## 4. Comparar com uma regra simples
Uma referência honesta: quanto o modelo acerta a mais do que chutar sempre a classe mais comum? Por isso olhamos também o F1 macro, que dá o mesmo peso às três classes.

In [ ]:
mais_comum = y_treino.value_counts().idxmax()
base = (y_teste == mais_comum).mean()
f1_base = f1_score(y_teste, [mais_comum] * len(y_teste), average="macro")
print(f"Sempre responder '{mais_comum}': acurácia {base:.3f}, F1 macro {f1_base:.3f}")
print(f"Nosso modelo: acurácia {acuracia:.3f}, F1 macro {f1_macro:.3f}")

## 5. Identificar os temas
Procuramos palavras-chave em cada avaliação. É um método simples e explicável, mas não entende contexto: "não me senti insegura" também conta como menção a segurança. Por isso, medimos **menções**, não fatos.

**Limites conferidos em uma amostra:**
- **Assédio:** muitas menções falam de vendedores ou taxistas insistentes. Este número não mede assédio sexual.
- **Iluminação:** muitas menções falam da luz do quarto ou do corredor, não da rua.

In [ ]:
def normalizar(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))

TEMAS = {
    "Segurança": r"\b(?:seguranca|segura|seguro|inseguranca|insegura|inseguro|perigos[oa]|assalt\w*|roub\w*|furt\w*)\b",
    "Assédio (qualquer tipo)": r"\b(?:assedi\w*)\b",
    "Iluminação (inclui quartos)": r"\b(?:iluminad[oa]s?|iluminacao|escur[oa]s?)\b",
    "Localização": r"\b(?:localizacao|localizad[oa]|bairro|regiao|proximo|perto)\b",
    "Atendimento": r"\b(?:atendimento|funcionari\w*|recepcao|recepcionista\w*|equipe)\b",
    "Acolhimento": r"\b(?:acolhedor\w*|acolhid[oa]s?|acolhimento|receptiv\w*|atencios\w*|me senti em casa)\b",
}

texto_norm = df["texto"].map(normalizar)
for tema, padrao in TEMAS.items():
    df[tema] = texto_norm.str.contains(padrao, regex=True)

df["Viajou sozinha"] = texto_norm.str.contains(
    r"\b(?:viajei|viajando|viajo|fui|estava|fiquei|hospedei[- ]?me|me hospedei|mulher|sendo mulher)\b[^.!?]{0,30}\bsozinha\b", regex=True)

tabela = []
for tema in list(TEMAS) + ["Viajou sozinha"]:
    parte = df[df[tema]]
    tabela.append({"tema": tema, "avaliacoes": int(len(parte)),
                   "percentual": round(100 * len(parte) / len(df), 2),
                   "nota_media": round(float(parte["nota"].mean()), 2),
                   "negativas_pct": round(100 * float((parte["sentimento"] == "negativo").mean()), 1)})
temas = pd.DataFrame(tabela).sort_values("avaliacoes", ascending=False)
temas

## 6. Olhar quem diz que viajou sozinha
Este é um recorte aproximado. A base não informa o gênero, então usamos só as frases com "sozinha". Isso não cobre todas as mulheres, e alguns textos falam de outra pessoa, como uma funcionária. Em uma amostra de 16 textos lidos, 13 eram de fato sobre a própria viajante, então trate os números como indicativos.

In [ ]:
sozinhas = df[df["Viajou sozinha"]]
geral = {"grupo": "Todas as avaliações", "avaliacoes": len(df), "nota_media": df["nota"].mean(),
         "negativas_pct": 100 * (df["sentimento"] == "negativo").mean(), "seguranca_pct": 100 * df["Segurança"].mean()}
recorte = {"grupo": "Mencionam viagem sozinha", "avaliacoes": len(sozinhas), "nota_media": sozinhas["nota"].mean(),
           "negativas_pct": 100 * (sozinhas["sentimento"] == "negativo").mean(), "seguranca_pct": 100 * sozinhas["Segurança"].mean()}
comparacao = pd.DataFrame([geral, recorte]).round(2)
comparacao

## 7. Ver um gráfico dos temas

In [ ]:
import matplotlib.pyplot as plt

ordem = temas.sort_values("avaliacoes")
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(ordem["tema"], ordem["percentual"], color="#955439")
ax.set_xlabel("% das avaliações que mencionam o tema")
ax.set_title("Temas nas avaliações de hotéis")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 8. Salvar os arquivos do site
- `modelo_sentimento.json`: as palavras com mais peso, para o site classificar um comentário no navegador. Como usa só parte do modelo, o resultado no site é uma versão simplificada.
- `resumo_avaliacoes.json`: somente números agregados. Nenhum comentário original é copiado.

In [ ]:
palavras = np.array(vetor.get_feature_names_out())
pesos = {}
LIMITE = 2500
indices = set()
for i in range(len(classes)):
    indices.update(np.argsort(-np.abs(modelo.coef_[i]))[:LIMITE].tolist())
indices = sorted(indices)
saida_modelo = {
    "classes": classes,
    "intercepto": [round(float(v), 4) for v in modelo.intercept_],
    "pesos": {str(palavras[j]): [round(float(modelo.coef_[i][j]), 3) for i in range(len(classes))] for j in indices},
}
with open("modelo_sentimento.json", "w", encoding="utf-8") as f:
    json.dump(saida_modelo, f, ensure_ascii=False, separators=(",", ":"))

resumo = {
    "fonte": "Brazilian Portuguese Hotel's Reviews Corpus (Mendeley Data, CC BY 4.0)",
    "avaliacoes_originais": len(comentarios),
    "avaliacoes_validas": int(len(df)),
    "periodo": [int(df["ano"].min()), int(df["ano"].max())],
    "acuracia": round(float(acuracia), 3),
    "f1_macro": round(float(f1_macro), 3),
    "acuracia_base": round(float(base), 3),
    "f1_macro_base": round(float(f1_base), 3),
    "sentimentos_pct": {k: round(100 * float(v), 1) for k, v in df["sentimento"].value_counts(normalize=True).items()},
    "temas": temas.to_dict(orient="records"),
    "comparacao": comparacao.to_dict(orient="records"),
}
with open("resumo_avaliacoes.json", "w", encoding="utf-8") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=1)

print(json.dumps(resumo, ensure_ascii=False, indent=1))
try:
    from google.colab import files
    files.download("modelo_sentimento.json")
    files.download("resumo_avaliacoes.json")
except ImportError:
    print("Fora do Colab: arquivos salvos na pasta atual.")